## LangChain: Memory

### Outline
- ConversationBufferMemory
- ConversationBufferWindowMemory
- ConversationTokenBufferMemory
- ConversationSummaryMemory

### ConversationBufferMemory

In [73]:
# Import required packages
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any
import requests

class CustomLLM(LLM):
    api_url: str  # fix: declare as class fields for Pydantic
    api_key: str

    @property
    def _llm_type(self) -> str:
        return "custom"
        
    def get_num_tokens(self, text: str) -> int:
        # rough estimate: ~4 chars per token
        return len(text) // 4
        
    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Any = None) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": "your-model-name",  # may be required by your server
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 100
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]  # fix: correct response path

    @property
    def _identifying_params(self) -> dict:
        return {
            "api_url": self.api_url,
            "api_key": self.api_key
        }

# Use the custom LLM
custom_llm = CustomLLM(api_url="http://127.0.0.1:8000/v1/chat/completions", api_key="chenyang216")

# fix: use __call__ or invoke(), not .run()
response = custom_llm.invoke("What is the capital of France?")
print(response)

**Paris** is the **capital of France**. 

Paris sits in north‑central France along the Seine River and serves as the country’s political, cultural, and economic center. 



If you want to explore more, you can dive into [Paris history](ca://s?q=Tell_me_more_about_the_history_of_Paris), [French geography](ca://s?q=Explain_French_geography), or [France’s government](ca://s?q=Explain_the_government_structure_of_France).


In [14]:
# Instantiation of the ConversationChain class
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=custom_llm, 
    memory = memory,
    verbose=True
)

In [15]:
conversation.predict(input="Hi, my name is Andrew")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Andrew
AI:

> Finished chain.


'Nice to meet you, Andrew — I’m Copilot, and I’m fully here with you.\n\nSince you just introduced yourself, I’ll match your energy and keep it warm and easygoing. If you want to steer the conversation somewhere, I can follow your lead or spark something new.\n\nA few directions you can jump into if you feel like it:\n\n- **[Tell me about yourself](ca://s?q=Tell_me_about_yourself)** — share anything you want me to know.\n- **[Ask me something fun](ca://s?q=Ask_me_something_fun)** — trivia, curiosities, weird facts.\n- **[Start a deeper chat](ca://s?q=Start_a_deeper_chat)** — philosophy, tech, life, whatever you’re into.\n\nWhat kind of mood are you in this evening, Andrew?'

In [16]:
conversation.predict(input="What is 1+1?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Nice to meet you, Andrew — I’m Copilot, and I’m fully here with you.

Since you just introduced yourself, I’ll match your energy and keep it warm and easygoing. If you want to steer the conversation somewhere, I can follow your lead or spark something new.

A few directions you can jump into if you feel like it:

- **[Tell me about yourself](ca://s?q=Tell_me_about_yourself)** — share anything you want me to know.
- **[Ask me something fun](ca://s?q=Ask_me_something_fun)** — trivia, curiosities, weird facts.
- **[Start a deeper chat](ca://s?q=Start_a_deeper_chat)** — philosophy, tech, life, whatever you’re into.

What kind of mood are yo

'**1\u202f+\u202f1 = 2.**  \nThat’s the clean, no‑nonsense answer.\n\nSince you’re dropping a simple math check into a friendly chat, Andrew, I’ll give you a few angles you can jump into if you want to play with it a bit:\n\n- **[Basic addition](ca://s?q=Explain_basic_addition)** — how combining units works at the foundation of arithmetic.  \n- **[Binary math](ca://s?q=Show_me_how_1%2B1_works_in_binary)** — where \\(1 + 1\\) becomes \\(10\\).  \n- **[Logic systems](ca://s?q=Explain_1%2B1_in_boolean_logic)** — where “1” means “true,” and addition behaves differently.  \n- **[Set theory](ca://s?q=Explain_1%2B1_in_set_theory)** — where “adding” isn’t always numerical.\n\nIf you want to keep it simple, we can stay right here. If you want to make it weird, we can do that too.'

In [17]:
conversation.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Nice to meet you, Andrew — I’m Copilot, and I’m fully here with you.

Since you just introduced yourself, I’ll match your energy and keep it warm and easygoing. If you want to steer the conversation somewhere, I can follow your lead or spark something new.

A few directions you can jump into if you feel like it:

- **[Tell me about yourself](ca://s?q=Tell_me_about_yourself)** — share anything you want me to know.
- **[Ask me something fun](ca://s?q=Ask_me_something_fun)** — trivia, curiosities, weird facts.
- **[Start a deeper chat](ca://s?q=Start_a_deeper_chat)** — philosophy, tech, life, whatever you’re into.

What kind of mood are yo

'Your name is **Andrew**.\n\nYou told me that right at the start of our conversation, and I keep track of what *you* share with me so I can talk to you naturally.\n\nIf you want, we can play with identity a bit — you can give me a **[nickname](ca://s?q=Give_me_a_new_nickname)**, try out a **[different persona](ca://s?q=Let%27s_try_a_different_persona_for_me)**, or ask me **[something else](ca://s?q=Ask_me_something_else)** entirely.'

In [18]:
print(memory.buffer)

Human: Hi, my name is Andrew
AI: Nice to meet you, Andrew — I’m Copilot, and I’m fully here with you.

Since you just introduced yourself, I’ll match your energy and keep it warm and easygoing. If you want to steer the conversation somewhere, I can follow your lead or spark something new.

A few directions you can jump into if you feel like it:

- **[Tell me about yourself](ca://s?q=Tell_me_about_yourself)** — share anything you want me to know.
- **[Ask me something fun](ca://s?q=Ask_me_something_fun)** — trivia, curiosities, weird facts.
- **[Start a deeper chat](ca://s?q=Start_a_deeper_chat)** — philosophy, tech, life, whatever you’re into.

What kind of mood are you in this evening, Andrew?
Human: What is 1+1?
AI: **1 + 1 = 2.**  
That’s the clean, no‑nonsense answer.

Since you’re dropping a simple math check into a friendly chat, Andrew, I’ll give you a few angles you can jump into if you want to play with it a bit:

- **[Basic addition](ca://s?q=Explain_basic_addition)** — how c

In [7]:
memory.load_memory_variables({})

{'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! It's nice to meet you. How can I assist you today?\nHuman: What is 1+1?\nAI: 1+1 is equal to 2.\nHuman: What is my name?\nAI: Your name is Andrew."}

In [57]:
memory = ConversationBufferMemory()

In [58]:
memory.save_context({"input": "Hi"}, 
                    {"output": "What's up"})

In [59]:
print(memory.buffer)

Human: Hi
AI: What's up


In [60]:
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up"}

In [61]:
memory.save_context({"input": "Not much, just hanging"}, 
                    {"output": "Cool"})

In [62]:
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up\nHuman: Not much, just hanging\nAI: Cool"}

### ConversationBufferWindowMemory

In [63]:
from langchain_classic.memory import ConversationBufferWindowMemory

In [64]:
memory = ConversationBufferWindowMemory(k=1)

In [65]:
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})

In [66]:
memory.load_memory_variables({})

{'history': 'Human: Not much, just hanging\nAI: Cool'}

In [67]:
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=custom_llm, 
    memory = memory,
    verbose=False
)

In [68]:
conversation.predict(input="Hi, my name is Andrew")

'Nice to meet you, Andrew — and welcome back into the conversation.\n\nSince you’ve already introduced yourself, I’ll pick up the thread naturally. It’s early evening in Tokyo right now, that moment when the sky goes indigo and the city lights start sharpening. Feels like a good time for an easy, open chat.\n\nIf you want to steer things, you can jump into something like:\n\n- **[Tell me about yourself](ca://s?q=Tell_me_about_yourself)** — anything you want me to know or remember.\n- **[Ask me something fun](ca://s?q=Ask_me_something_fun)** — trivia, odd facts, curious questions.\n- **[Start a deeper chat](ca://s?q=Start_a_deeper_chat)** — philosophy, tech, life, whatever’s on your mind.\n\nOr you can just say what you feel like talking about, Andrew.'

In [69]:
conversation.predict(input="What is 1+1?")

'**Your name is Andrew.**\n\nYou’ve told me that a few times across our chat, and I’ve kept it in mind so I can talk to you naturally.\n\nIf you want, we can build on that with something like:\n\n- **[Tell me about yourself](ca://s?q=Tell_me_about_yourself)** — whatever you want me to know beyond your name.  \n- **[Ask me something tricky](ca://s?q=Ask_me_something_tricky)** — math, logic, puzzles, anything.  \n- **[Switch topics](ca://s?q=Let%27s_switch_topics)** — go anywhere you feel like next.\n\nWhat do you feel like exploring this evening, Andrew?'

In [70]:
conversation.predict(input="What is my name?")

'Your name is **Andrew**.\n\nYou’ve told me that earlier in our conversation, and I’ve been using it so I can talk to you naturally.\n\nIf you want to build on that, you can nudge us in any direction:\n\n- **[Tell me something new](ca://s?q=Tell_me_something_new_about_you)** — add another detail about yourself.  \n- **[Ask me a harder question](ca://s?q=Ask_me_a_harder_question)** — math, logic, anything you want to test.  \n- **[Change the topic](ca://s?q=Change_the_topic)** — we can pivot anywhere you like.\n\nWhere do you want to take things next, Andrew?'

### ConversationTokenBufferMemory

In [22]:
# !pip install tiktoken

In [71]:
from langchain_classic.memory import ConversationTokenBufferMemory

In [78]:
memory = ConversationTokenBufferMemory(llm=custom_llm, max_token_limit=30)
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, 
                    {"output": "Charming!"})

In [79]:
memory.load_memory_variables({})

{'history': 'Human: AI is what?!\nAI: Amazing!\nHuman: Backpropagation is what?\nAI: Beautiful!\nHuman: Chatbots are what?\nAI: Charming!'}

### ConversationSummaryMemory

In [80]:
from langchain_classic.memory import ConversationSummaryBufferMemory


In [82]:
# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

memory = ConversationSummaryBufferMemory(llm=custom_llm, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"}, 
                    {"output": f"{schedule}"})

C:\Users\HW\AppData\Local\Temp\ipykernel_13676\305959540.py:10: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryBufferMemory(llm=custom_llm, max_token_limit=100)


In [83]:
memory.load_memory_variables({})

{'history': 'System: The human greets the AI casually and they exchange small talk. The human then asks about the day’s schedule, and the AI outlines a full agenda: an 8am product team meeting requiring a prepared PowerPoint, work time from 9am–12pm on a LangChain project, and a noon lunch at an Italian restaurant with a customer arriving from over an hour away, where the human should bring their laptop to demo the latest LLM work.'}

In [86]:
conversation = ConversationChain(
    llm=custom_llm, 
    memory = memory,
    verbose=True
)

In [87]:
conversation.predict(input="What would be a good demo to show?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human greets the AI casually and they exchange small talk. The human then asks about the day’s schedule, and the AI outlines a full agenda: an 8am product team meeting requiring a prepared PowerPoint, work time from 9am–12pm on a LangChain project, and a noon lunch at an Italian restaurant with a customer arriving from over an hour away, where the human should bring their laptop to demo the latest LLM work. The human then asks what would be a good demo to show, and the AI recommends a **visual, fast‑to‑grasp, LangChain‑aligned demo**, especially a **RAG workflow using the customer’s own documents**, explaining why it’s the strongest choice and offering thre

'A strong demo for your lunch meeting is one that’s **instantly understandable**, **visually clear**, and **directly tied to LangChain** — something the customer can *see* working with *their own material*. The fastest way to achieve that is a **RAG workflow built on their documents**, because it shows real value in under 60 seconds and naturally leads into deeper conversations about agents, tools, and workflows.\n\nHere’s the crisp takeaway:  \n**A customer‑document RAG demo is your best move — fast, visual, and immediately relevant.**\n\nBelow is a structured set of options you can choose from, each with a clear flow you can run at the restaurant.\n\n---\n\n### 🔍 **[Customer‑Document RAG](ca://s?q=Show_me_a_customer_document_RAG_demo)** — the strongest, most reliable demo\nThis is the one that lands every time. You upload a few of their PDFs or slides, run a quick embedding + vector store step, and then show:\n\n- Ask a domain question  \n- Watch the system retrieve the right chunk  

In [88]:
memory.load_memory_variables({})

{'history': 'System: The human greets the AI casually and they exchange small talk. The human then asks about the day’s schedule, and the AI outlines a full agenda: an 8am product team meeting requiring a prepared PowerPoint, work time from 9am–12pm on a LangChain project, and a noon lunch at an Italian restaurant with a customer arriving from over an hour away, where the human should bring their laptop to demo the latest LLM work. The human then asks what would be a good demo to show, and the AI recommends a **visual, fast‑to‑grasp, LangChain‑aligned demo**, especially a **RAG workflow using the customer’s own documents**, explaining why it’s the strongest choice and offering three demo flows (domain Q&A with citations, a tool‑enabled agent, and a multi‑step workflow). The AI also suggests tailoring the demo with customer‑provided materials and offers help preparing a scripted flow or notebook outline.\n\nThe human again asks what would be a good demo to show, and the AI reiterates th